In [ ]:
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI

# CONFIG

INPUT_PATH = Path(r"D:\Experiments\Hiberno_English_Dictonary\data\hiberno_dictionary_paragraphs.json")
OUTPUT_PATH = Path("outputs_GPT4all.jsonl")   # append-only
ERROR_PATH = Path("errors_GPT4all.jsonl")

# GPT4All local API server
BASE_URL = "http://localhost:4891/v1"
MODEL_NAME = "Reasoner v1"

# QA / RUN CONTROL
MAX_NEW_ROWS = 10
SKIP_UNFINISHED_ROWS = 0
MAX_TOTAL_ROWS_SCANNED = None

# Generation settings
TEMPERATURE = 0.1
MAX_TOKENS = 1200

# CLIENT

client = OpenAI(
    base_url=BASE_URL,
)

# LOAD INPUT

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data).rename(
    columns={"paragraph_id": "entry_id", "text": "entry_text"}
)

# Optional guard
if MAX_TOTAL_ROWS_SCANNED is not None:
    df = df.head(MAX_TOTAL_ROWS_SCANNED)

# LOAD COMPLETED IDS

done_ids = set()
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done_ids.add(str(obj["entry_id"]))
            except Exception:
                pass

print("Already done:", len(done_ids))

# PROMPT

def build_prompt(text: str) -> str:
    return f"""
Return JSON only.

Extract exactly these keys:
headword_raw
headword
pronunciations
part_of_speech
definition
examples
etymology
cross_references
region_mentions

Rules:
- Output must be valid JSON object only.
- No markdown.
- No commentary.
- lists must be arrays
- use [] if empty
- use "" if a scalar field is empty
- examples must always be a JSON array
- pronunciations must always be a JSON array
- cross_references must always be a JSON array
- region_mentions must always be a JSON array

ENTRY:
{text}
""".strip()

# SAFE JSON PARSE

def safe_parse(text: str):
    try:
        return json.loads(text), ""
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(text[start:end + 1]), ""
            except Exception as e:
                return None, str(e)
        return None, "no json found"

# NORMALIZATION

def normalize_parsed(obj: dict) -> dict:
    expected_list_fields = [
        "pronunciations",
        "examples",
        "cross_references",
        "region_mentions",
    ]
    expected_scalar_fields = [
        "headword_raw",
        "headword",
        "part_of_speech",
        "definition",
        "etymology",
    ]

    cleaned = {}

    for key in expected_scalar_fields:
        value = obj.get(key, "")
        if value is None:
            value = ""
        elif not isinstance(value, str):
            value = str(value)
        cleaned[key] = value.strip()

    for key in expected_list_fields:
        value = obj.get(key, [])
        if value is None:
            value = []
        elif not isinstance(value, list):
            value = [value]
        cleaned[key] = value

    return cleaned

# VALIDATION

def validate(obj: dict):
    flags = {
        "missing_headword": False,
        "missing_definition": False,
        "bad_examples": False,
    }

    if not obj.get("headword"):
        flags["missing_headword"] = True

    if not obj.get("definition"):
        flags["missing_definition"] = True

    if not isinstance(obj.get("examples", []), list):
        flags["bad_examples"] = True

    needs_review = any(flags.values())
    return flags, needs_review

#  CALL

def ask_model(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an information extraction engine. "
                    "Return strict JSON only."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    return response.choices[0].message.content

# MAIN LOOP

processed_new = 0
seen_unfinished = 0

with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f, \
     open(ERROR_PATH, "a", encoding="utf-8") as err_f:

    for i, row in df.iterrows():
        entry_id = str(row["entry_id"])
        entry_text = row["entry_text"]

        if entry_id in done_ids:
            continue

        seen_unfinished += 1

        if seen_unfinished <= SKIP_UNFINISHED_ROWS:
            continue

        if MAX_NEW_ROWS is not None and processed_new >= MAX_NEW_ROWS:
            break

        print(f"Processing {entry_id} | batch item {processed_new + 1}")

        try:
            prompt = build_prompt(entry_text)
            raw = ask_model(prompt)

            parsed, parse_err = safe_parse(raw)

            if parsed is None:
                err_f.write(json.dumps({
                    "entry_id": entry_id,
                    "error": parse_err,
                    "raw": raw
                }, ensure_ascii=False) + "\n")
                err_f.flush()
                continue

            parsed = normalize_parsed(parsed)
            flags, needs_review = validate(parsed)

            record = {
                "entry_id": entry_id,
                "source_text": entry_text,
                "data": parsed,
                "flags": flags,
                "needs_review": needs_review
            }

            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_f.flush()

            processed_new += 1

        except Exception as e:
            err_f.write(json.dumps({
                "entry_id": entry_id,
                "error": str(e)
            }, ensure_ascii=False) + "\n")
            err_f.flush()

print("Done.")
print("New rows processed:", processed_new)

Already done: 1
Processing 1 | batch item 1
Processing 2 | batch item 2
Processing 3 | batch item 3
Processing 4 | batch item 4
Processing 5 | batch item 5
Processing 6 | batch item 6
Processing 7 | batch item 7
Processing 8 | batch item 8
Processing 9 | batch item 9
Processing 10 | batch item 10
Done.
New rows processed: 10
